In [ ]:
"""
frame_alignment_vid.ipynb

generate a video to confirm that the alignment is correct

Author: Stellina X. Ao
Created: 2026-11-11
Last Modified: 2026-11-11
Python Version: 3.11.14
"""

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
from core.data import default_sessions

subj_id = "MR94"
sess_id = default_sessions[subj_id]

In [ ]:
from sg.models import make_tre, Encoder

encoder_svd = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    add_svd=True,
    num_svd=20,
    stepsize_s=0.1,
)
encoder_svd.verify()

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    add_svd=False,
    stepsize_s=0.1,
)
encoder.verify()

### alignment

In [ ]:
from utils.paths import DATA_DIR
import cv2

vid_path = (
    DATA_DIR
    / subj_id
    / sess_id
    / "DynamicForaging"
    / f"{subj_id}_DynamicForaging_{sess_id}_cam1_run000_00000000.avi"
)

cap = cv2.VideoCapture(str(vid_path))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(n_frames)

In [ ]:
import cv2


def put_text(frame, text, org, scale, color, thickness):
    font = cv2.FONT_HERSHEY_DUPLEX
    cv2.putText(frame, text, org, font, scale, (0, 0, 0), thickness + 2, cv2.LINE_AA)
    cv2.putText(frame, text, org, font, scale, color, thickness, cv2.LINE_AA)


def write_trial_video(
    vid_path,
    frame_trial_idxs,
    trial_idxs,
    out_path,
    fourcc="mp4v",
    flash_s=0.25,
    flash_color=(0, 0, 255),  # BGR
    text_color=(255, 255, 255),
    font_scale=1.0,
    thickness=2,
):
    cap = cv2.VideoCapture(str(vid_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(
        str(out_path), cv2.VideoWriter_fourcc(*fourcc), fps, (w, h)
    )

    flash_frames = max(1, round(flash_s * fps))
    border = max(4, w // 80)

    try:
        for t in trial_idxs:
            start, end = (int(i) for i in frame_trial_idxs[t])
            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
            for f in range(start, end):
                ok, frame = cap.read()
                if not ok:
                    print(f"trial {t}: read failed at frame {f}")
                    break

                if f - start < flash_frames:
                    # rectangle is centred on the frame edge, so double the thickness
                    cv2.rectangle(
                        frame, (0, 0), (w - 1, h - 1), flash_color, 2 * border
                    )

                put_text(
                    frame,
                    f"trial {t}   {(f - start) / fps:.2f} s",
                    (15, int(40 * font_scale)),
                    font_scale,
                    text_color,
                    thickness,
                )
                writer.write(frame)
    finally:
        cap.release()
        writer.release()

In [ ]:
import numpy as np
from utils.paths import DATA_DIR
from core.data import get_event_trial_idxs

trial_data = encoder.trial_data
session_data = encoder.session_data
svd_stepsize = None


trial_start = session_data["events"]["event_timestamps"][0]
movie_frame = session_data["events"]["event_timestamps"][1]

movements = np.load(
    DATA_DIR
    / subj_id
    / sess_id
    / "DynamicForaging"
    / f"{subj_id}_DynamicForaging_{sess_id}_cam1_run000_00000000.npy",
    allow_pickle=True,
).item()

frame_trial_idxs = get_event_trial_idxs(
    trial_start,
    trial_data,
    movie_frame,
    svd_stepsize=None,
    tpre=0.5,
    tpost=1.0,
    alignment="choice",
    full_trial=False,
)

In [ ]:
len(trial_start), len(trial_data)

In [ ]:
trial_start[0] + trial_data["response_time"].iloc[0]

In [ ]:
frame_trial_idxs[0]

In [ ]:
session_data

In [ ]:
frame_trial_idxs

In [ ]:
import cv2
from utils.paths import FIGURES_DIR

out_path = FIGURES_DIR / "svds" / "explore" / "spout_movement_anticipatory.mp4"
fourcc = "mp4v"
trial_idxs = [20, 22, 54, 67, 101]  # first three and last three

write_trial_video(
    vid_path=vid_path,
    frame_trial_idxs=frame_trial_idxs,
    trial_idxs=trial_idxs,
    out_path=out_path,
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure


def plot_activity(
    rates,  # [n_bins, n_neurons]
    t,
    ax=None,
    legend=False,
    xlim=None,
    ylim=None,
    xlabel="time in trial (s)",
    ylabel="activity (z)",
    line_kwargs=None,
):
    if ax is None:
        fig, ax = plt.subplots(figsize=(2.5, 2.5), tight_layout=True)
    else:
        fig = ax.get_figure()

    line_kwargs = {
        "neurons": dict(color="0.7", lw=0.5),
        "mean": dict(color="C3", lw=2),
        **(line_kwargs or {}),
    }
    ax.plot(t, rates, **line_kwargs["neurons"])[0].set_label("neurons")
    ax.plot(t, rates.mean(1), label="mean", **line_kwargs["mean"])
    ax.set(xlim=xlim, ylim=ylim, xlabel=xlabel, ylabel=ylabel)
    if legend:
        ax.legend(frameon=False)
    return fig, ax


def write_trial_video(
    vid_path,
    frame_trial_idxs,
    trial_idxs,
    rates,  # [n_trials * n_bins, n_neurons]
    out_path,
    bin_s=None,  # None: bins evenly span each trial's frames
    zscore=True,
    panel_frac=1.0,  # panel width as a fraction of the frame width
    dpi=100,
    cursor_color=(0, 0, 0),  # BGR
    line_kwargs=None,
    fourcc="mp4v",
    flash_s=0.25,
    flash_color=(0, 0, 255),  # BGR
    text_color=(255, 255, 255),
    font_scale=1.0,
    thickness=2,
):
    rates = np.asarray(rates, dtype=float)
    n_bins, rem = divmod(len(rates), len(frame_trial_idxs))
    assert rem == 0, "rates must have n_trials * n_bins rows"
    if zscore:
        # per neuron over the whole recording, so scale is comparable across trials
        rates = (rates - rates.mean(0)) / (rates.std(0) + 1e-9)
    ylim = tuple(np.percentile(rates, [0.5, 99.5]))

    cap = cv2.VideoCapture(str(vid_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    panel_w = int(w * panel_frac) // 2 * 2
    writer = cv2.VideoWriter(
        str(out_path), cv2.VideoWriter_fourcc(*fourcc), fps, (w + panel_w, h)
    )

    fig = Figure(figsize=(panel_w / dpi, h / dpi), dpi=dpi, layout="tight")
    canvas = FigureCanvasAgg(fig)
    ax = fig.subplots()

    flash_frames = max(1, round(flash_s * fps))
    border = max(4, w // 80)

    try:
        for t in trial_idxs:
            start, end = (int(i) for i in frame_trial_idxs[t])

            # render the static activity panel once per trial
            bs = bin_s or (end - start) / fps / n_bins
            dur = n_bins * bs
            ax.clear()
            plot_activity(
                rates[t * n_bins : (t + 1) * n_bins],
                (np.arange(n_bins) + 0.5) * bs,
                ax=ax,
                xlim=(0, dur),
                ylim=ylim,
                line_kwargs=line_kwargs,
            )
            canvas.draw()
            base = cv2.cvtColor(np.asarray(canvas.buffer_rgba()), cv2.COLOR_RGBA2BGR)
            if base.shape[:2] != (h, panel_w):
                base = cv2.resize(base, (panel_w, h))
            y0, y1 = int(h - ax.bbox.y1), int(h - ax.bbox.y0)  # image rows

            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
            for f in range(start, end):
                ok, frame = cap.read()
                if not ok:
                    print(f"trial {t}: read failed at frame {f}")
                    break

                if f - start < flash_frames:
                    # rectangle is centred on the frame edge, so double the thickness
                    cv2.rectangle(
                        frame, (0, 0), (w - 1, h - 1), flash_color, 2 * border
                    )

                put_text(
                    frame,
                    f"trial {t}   {(f - start) / fps:.2f} s",
                    (15, int(40 * font_scale)),
                    font_scale,
                    text_color,
                    thickness,
                )

                panel = base.copy()
                x = int(ax.transData.transform((min((f - start) / fps, dur), 0))[0])
                cv2.line(panel, (x, y0), (x, y1), cursor_color, 2, cv2.LINE_AA)
                writer.write(np.hstack([frame, panel]))
    finally:
        cap.release()
        writer.release()

In [ ]:
encoder.view_peths(reg="ACC_M2")

In [ ]:
encoder.reg_idxs["ACC_M2"][2]

In [ ]:
from scipy.stats import sem

robs_3d = encoder.robs.reshape(
    (encoder.num_trials, encoder.num_bins, encoder.num_units)
)

l_mask = encoder.trial_data["response"] == 1
r_mask = encoder.trial_data["response"] == -1

peth_l = robs_3d[l_mask, :, 99]
peth_r = robs_3d[r_mask, :, 99]

l_m = peth_l.mean(axis=0)
r_m = peth_r.mean(axis=0)

l_s = sem(peth_l, axis=0)
r_s = sem(peth_r, axis=0)

plt.figure()
plt.plot(l_m, label="l")
plt.plot(r_m, label="r")
plt.fill_between(range(encoder.num_bins), l_m - l_s, l_m + l_s, alpha=0.5)
plt.fill_between(range(encoder.num_bins), r_m - r_s, r_m + r_s, alpha=0.5)
plt.legend()
plt.show()

In [ ]:
encoder.view_peths(reg="ACC_M2")

In [ ]:
encoder.view_weights(reg="ACC_M2")

In [ ]:
A = robs_3d.mean(axis=0)
peak_sort_idxs = np.argsort(np.argmax(A, axis=0))

plt.figure()
plt.imshow(A[:, peak_sort_idxs].T, aspect="auto")
plt.colorbar()
plt.show()

In [ ]:
import cv2
from utils.paths import FIGURES_DIR

out_path = FIGURES_DIR / "svds" / "explore" / "activty_alignment.mp4"
fourcc = "mp4v"

write_trial_video(
    vid_path=vid_path,
    frame_trial_idxs=frame_trial_idxs,
    trial_idxs=trial_idxs,
    rates=encoder.robs,
    out_path=out_path,
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np


def write_trial_video(
    vid_path,
    frame_trial_idxs,
    trial_idxs,
    rates,  # [n_trials, n_bins, n_neurons]
    out_path,
    bin_s=None,  # None: bins evenly span each trial's frames
    zscore=True,
    cmap="viridis",
    vlim=None,  # (vmin, vmax); None: 1st-99th percentile of rates
    strip_w=40,  # width of the neuron strip in pixels
    fourcc="mp4v",
    flash_s=0.25,
    flash_color=(0, 0, 255),  # BGR
    text_color=(255, 255, 255),
    font_scale=1.0,
    thickness=2,
):
    rates = np.asarray(rates, dtype=float)
    assert rates.ndim == 3 and len(rates) == len(frame_trial_idxs)
    n_bins = rates.shape[1]
    if zscore:
        # per neuron over all trials and bins, so scale is comparable across trials
        rates = (rates - rates.mean((0, 1))) / (rates.std((0, 1)) + 1e-9)
    vmin, vmax = vlim if vlim is not None else np.percentile(rates, [1, 99])
    lut = (plt.get_cmap(cmap)(np.linspace(0, 1, 256))[:, 2::-1] * 255).astype(np.uint8)
    lut_idx = (np.clip((rates - vmin) / (vmax - vmin + 1e-9), 0, 1) * 255).astype(
        np.uint8
    )

    cap = cv2.VideoCapture(str(vid_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(
        str(out_path), cv2.VideoWriter_fourcc(*fourcc), fps, (w + strip_w, h)
    )

    flash_frames = max(1, round(flash_s * fps))
    border = max(4, w // 80)

    try:
        for t in trial_idxs:
            start, end = (int(i) for i in frame_trial_idxs[t])
            bs = bin_s or (end - start) / fps / n_bins
            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
            for f in range(start, end):
                ok, frame = cap.read()
                if not ok:
                    print(f"trial {t}: read failed at frame {f}")
                    break

                if f - start < flash_frames:
                    # rectangle is centred on the frame edge, so double the thickness
                    cv2.rectangle(
                        frame, (0, 0), (w - 1, h - 1), flash_color, 2 * border
                    )

                put_text(
                    frame,
                    f"trial {t}   {(f - start) / fps:.2f} s",
                    (15, int(40 * font_scale)),
                    font_scale,
                    text_color,
                    thickness,
                )

                # [n_neurons, 1, 3] column of colours for the current bin, neuron 0 at top
                b = min(int((f - start) / fps / bs), n_bins - 1)
                strip = cv2.resize(
                    lut[lut_idx[t, b]][:, None],
                    (strip_w, h),
                    interpolation=cv2.INTER_NEAREST,
                )
                writer.write(np.hstack([frame, strip]))
    finally:
        cap.release()
        writer.release()

In [ ]:
import cv2
from utils.paths import FIGURES_DIR

out_path = FIGURES_DIR / "svds" / "explore" / "activty_alignment.mp4"
fourcc = "mp4v"

write_trial_video(
    vid_path=vid_path,
    frame_trial_idxs=frame_trial_idxs,
    trial_idxs=np.arange(1, encoder.num_trials, 10),
    rates=encoder.robs.reshape(
        (encoder.num_trials, encoder.num_bins, encoder.num_units)
    ),
    out_path=out_path,
)